In [1]:
pip install nltk


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import string
import re

import nltk
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer ,  TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report




In [3]:
nltk.download('stopwords') #step1: to download stopwords # or use lemmatizer 
#lemm = WordNetLemmatizer()
#df['clean_reviews'] = df['clean_reviews'].apply(
 #   lambda x: ' '.join([lemm.lemmatize(word) for word in x.split()])
#)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\my4le\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
from nltk.corpus import stopwords

In [5]:
df = pd.read_csv("IMDB Dataset.csv") #step 2
df.columns = ['review','sentiment']

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df['sentiment']=df['sentiment'].map({"negative":0,"positive":1}) #step 3

In [8]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [9]:
stemmer = PorterStemmer() #step 4
stop_words = set(stopwords.words('english'))

In [10]:
def clean_text(text): #step 5 Tokenization
    text = text.lower()
    text = re.sub(r'\W',' ',text) # \W remove punctuations
    text = re.sub(r'\s+',' ',text) # \s remove extra spaces
    words = text.split() #token
    words= [stemmer.stem(word) for word in words if word not in stop_words] # stemming and stopwords
    return ' '.join(words) # rejoin whole words

df["cleaned"] = df['review'].apply(clean_text)
df.head()

,review,sentiment,cleaned
0,One of the other reviewers has mentioned that ...,1,one review mention watch 1 oz episod hook righ...
1,A wonderful little production. <br /><br />The...,1,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,1,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,0,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter mattei love time money visual stun film...


In [11]:
# step 6 Feature extraction / Numeric Representation using NLP models

# Option A : Bag of Words

#vectorizer =  CountVectorizer()
#X1 = vectorizer.fit_transform(df['cleaned']).toarray()     #while using this code getting a memory err

#so using below code

#X1 = vectorizer.fit_transform(df['cleaned'])

#limiting vocab to reduce memory err


vectorizer = CountVectorizer(max_features=5000)
X1 = vectorizer.fit_transform(df['cleaned']).toarray()



In [12]:
# vectorizer.vocabulary_


In [13]:
#option B tf-idf (better for weightings)

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['cleaned']).toarray()
y= df['sentiment']

In [14]:
# Step 7 : split data -  BOW
X_train,X_test,y_train,y_test = train_test_split(X1,y,test_size = 0.2,random_state = 10)



In [15]:
from sklearn.linear_model import LogisticRegression #step8 : Building model
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

xg = xgb.XGBClassifier()
xg.fit(X_train,y_train)
y_pred_xg = xg.predict(X_test)

lr = LogisticRegression(max_iter=2000)
lr.fit(X_train,y_train)
y_pred_lr = lr.predict(X_test)

rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)

In [16]:
print(classification_report(y_test,y_pred_lr))

              precision    recall  f1-score   support

           0       0.88      0.87      0.87      5108
           1       0.86      0.87      0.87      4892

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [17]:
print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.85      0.84      0.85      5108
           1       0.84      0.85      0.84      4892

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [18]:
print(classification_report(y_test,y_pred_xg))

              precision    recall  f1-score   support

           0       0.88      0.84      0.86      5108
           1       0.84      0.88      0.86      4892

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [19]:
y_pred_lr = lr.predict(X)

In [20]:
df["predicted_Labels_TFIDF"] = y_pred_lr

In [21]:
df.head()

,review,sentiment,cleaned,predicted_Labels_TFIDF
0,One of the other reviewers has mentioned that ...,1,one review mention watch 1 oz episod hook righ...,1
1,A wonderful little production. <br /><br />The...,1,wonder littl product br br film techniqu unass...,1
2,I thought this was a wonderful way to spend ti...,1,thought wonder way spend time hot summer weeke...,0
3,Basically there's a family where a little boy ...,0,basic famili littl boy jake think zombi closet...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter mattei love time money visual stun film...,1


In [22]:
#save model 

import pickle
with open("IMDB_Reviews_model.pkl","wb")as f:
    pickle.dump(lr,f)

# saving vectorized model

with open("NLP_model.pkl",'wb') as f:
    pickle.dump(tfidf,f)

In [23]:
# installing gensim for word embeddings 

#!pip install gensim

In [24]:
import gensim 
from gensim.models import Word2Vec

In [25]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300') #1.5 gb of model

In [26]:
df= pd.read_csv("IMDB Dataset.csv")
df.columns = ["review","sentiment"]

In [27]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [28]:
from gensim.utils import simple_preprocess
def preprocess_text(text: str):
    # lowercases, tokenizes, removes punctuation/numbers by default
    return simple_preprocess(text)

In [29]:
def sentence_vector(tokens, wv, dim=300):
    vecs = [wv[w] for w in tokens if w in wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

In [30]:
X_tokens = df["review"].astype(str).map(preprocess_text)
X_vec =np.vstack([sentence_vector(toks, wv) for toks in X_tokens])

In [31]:
X_vec

array([[ 0.03491639,  0.02569429,  0.04602862, ..., -0.0642433 ,
         0.0404548 , -0.02217407],
       [ 0.05999595,  0.05363364,  0.00603614, ..., -0.05286418,
         0.03124065, -0.01044623],
       [ 0.04745925,  0.04472633,  0.02417902, ..., -0.06246927,
         0.03445824, -0.00790037],
       ...,
       [ 0.05976124,  0.03012987,  0.06848656, ..., -0.04535903,
         0.04123665, -0.00130325],
       [ 0.06396001,  0.04098415,  0.03723428, ..., -0.05993358,
         0.0268233 , -0.00993106],
       [ 0.07289689,  0.03411576,  0.01319155, ..., -0.02466557,
         0.03879882, -0.04883417]], dtype=float32)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

In [33]:
lr = LogisticRegression()
lr.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [34]:
y_pred = lr.predict(X_test)
print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4961
           1       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



In [35]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.81      0.81      0.81      4961
           1       0.81      0.81      0.81      5039

    accuracy                           0.81     10000
   macro avg       0.81      0.81      0.81     10000
weighted avg       0.81      0.81      0.81     10000



In [36]:
xg = xgb.XGBClassifier()
xg.fit(X_train,y_train)
y_pred_xg = xg.predict(X_test)
print(classification_report(y_test,y_pred_xg))

              precision    recall  f1-score   support

           0       0.84      0.85      0.84      4961
           1       0.85      0.84      0.85      5039

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



In [37]:
#Using on new text

In [38]:
def predict_label(text:str):
    toks = preprocess_text(text)
    vec = sentence_vector(toks,wv).reshape(1,-1)
    pred = lr.predict(vec)[0]
    return "negative" if pred == 0 else "positive"

In [39]:
predict_label("First of all, let's get a few things straight here: a) I AM an anime fan- always has been as a matter of fact (I used to watch Speed Racer all the time in Preschool). b) I DO like several B-Movies because they're hilarious. c) I like the Godzilla movies- a lot.<br /><br />Moving on, when the movie first comes on, it seems like it's going to be your usual B-movie, down to the crappy FX, but all a sudden- BOOM! the anime comes on! This is when the movie goes WWWAAAAAYYYYY downhill.<br /><br />The animation is VERY bad & cheap, even worse than what I remember from SPEED RACER, for crissakes! In fact, it's so cheap, one of the few scenes from the movie I ""vividly"" remember is when a bunch of kids run out of a school... & it's the same kids over & over again! The FX are terrible, too; the dinosaurs look worse than Godzilla. In addition, the transition to live action to animation is unorganized, the dialogue & voices(especially the English dub that I viewed) was horrid & I was begging my dad to take the tape out of the DVD/ VHS player; The only thing that kept me surviving was cracking out jokes & comments like the robots & Joel/Mike on MST3K (you pick the season). Honestly, this is the only way to barely enjoy this movie & survive it at the same time.<br /><br />Heck, I'm planning to show this to another fellow otaku pal of mine on Halloween for a B-Movie night. Because it's stupid, pretty painful to watch & unintentionally hilarious at the same time, I'm giving this movie a 3/10, an improvement from the 0.5/10 I was originally going to give it.<br /><br />(According to my grading scale: 3/10 means Pretty much both boring & bad. As fun as counting to three unless you find a way to make fun of it, then it will become as fun as counting to 15.)")

'negative'

In [40]:
predict_label("I sure would like to see a resurrection of a up dated Seahunt series with the tech they have today it would bring back the kid excitement in me.I grew up on black and white TV and Seahunt with Gunsmoke were my hero's every week.You have my vote for a comeback of a new sea hunt.We need a change of pace in TV and this would work for a world of under water adventure.Oh by the way thank you for an outlet like this to view many viewpoints about TV and the many movies.So any ole way I believe I've got what I wanna say.Would be nice to read some more plus points about sea hunt.If my rhymes would be 10 lines would you let me submit,or leave me out to be in doubt and have me to quit,If this is so then I must go so lets do it.")

'positive'